In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
# Configuración de la conexión
server = '10.0.0.129'  # Reemplaza con el nombre o dirección IP de tu servidor SQL Server
database = 'DB_GENERAL'  # Reemplaza con el nombre de tu base de datos
username = 'sa'  # Reemplaza con tu nombre de usuario
password = 'Essalud23**'  # Reemplaza con tu contraseña
driver = 'ODBC Driver 17 for SQL Server'
# Cadena de conexión
connection_string = f'mssql+pyodbc://{username}:{password}@{server}/{database}?driver={driver}'

In [3]:
try:
    # Establecer la conexión con SQLAlchemy
    engine = create_engine(connection_string)

    # Especificar el esquema
    schema = 'jcardenas'

    # Construir la consulta SQL
    query = f'SELECT CODIGO_DEPENDENCIA, DEPENDENCIA, SIGLAS, CODDEP_RESPONSABLE FROM {schema}.H_DEPENDENCIA'

    # Leer los resultados en un DataFrame de Pandas
    df = pd.read_sql(query, engine)

except Exception as e:
    print(f'Error de conexión: {e}')

finally:
    # No es necesario cerrar la conexión con SQLAlchemy
    pass

In [4]:
df

,CODIGO_DEPENDENCIA,DEPENDENCIA,SIGLAS,CODDEP_RESPONSABLE
0,1,SEGURO SOCIAL DE SALUD DEL PERÚ,ESSALUD,NaN
1,2,GERENCIA CENTRAL DE PLANEAMIENTO Y PRESUPUESTO,GCPP,NaN
2,3,OFICINA DE APOYO Y CONTROL,OAC-GCPP,2.0
3,4,GERENCIA DE PRESUPUESTO,GP-GCPP,2.0
4,5,SUB GERENCIA DE PROCESO PRESUPUESTAL,SGPP,4.0
...,...,...,...,...
318,320,UNIDAD DE CONTROL DE FILTRACIONES OSPE CORPORA...,UCF,40.0
319,321,ASUNTOS JUDICIALES - PROCESOS JUDICIALES,SISPROJ,175.0
320,322,ADQUISICION DE AMBULANCIAS URBANAS Y RURALES A...,SRCIAAUR-OCi,143.0
321,323,Concurso Publico N 7-2022-ESSALUD/GCL-1 y proc...,SRCICPPCR-OCI,143.0


In [5]:
def obtener_siglas_padre(row):
        siglas = row['SIGLAS']
        cod_dep_responsable = row['CODDEP_RESPONSABLE']

        while pd.notna(cod_dep_responsable):
            padre_row = df[df['CODIGO_DEPENDENCIA'] == cod_dep_responsable]
            if not padre_row.empty:
                siglas_padre = padre_row['SIGLAS'].iloc[0]
                siglas = f"{siglas}-{siglas_padre}"
                cod_dep_responsable = padre_row['CODDEP_RESPONSABLE'].iloc[0]
            else:
                break

        return siglas

# Crear la nueva columna SIGLAS_NUEVAS
df['SIGLAS_NUEVAS'] = df.apply(obtener_siglas_padre, axis=1)


In [6]:
df

,CODIGO_DEPENDENCIA,DEPENDENCIA,SIGLAS,CODDEP_RESPONSABLE,SIGLAS_NUEVAS
0,1,SEGURO SOCIAL DE SALUD DEL PERÚ,ESSALUD,NaN,ESSALUD
1,2,GERENCIA CENTRAL DE PLANEAMIENTO Y PRESUPUESTO,GCPP,NaN,GCPP
2,3,OFICINA DE APOYO Y CONTROL,OAC-GCPP,2.0,OAC-GCPP-GCPP
3,4,GERENCIA DE PRESUPUESTO,GP-GCPP,2.0,GP-GCPP-GCPP
4,5,SUB GERENCIA DE PROCESO PRESUPUESTAL,SGPP,4.0,SGPP-GP-GCPP-GCPP
...,...,...,...,...,...
318,320,UNIDAD DE CONTROL DE FILTRACIONES OSPE CORPORA...,UCF,40.0,UCF-GCSPE
319,321,ASUNTOS JUDICIALES - PROCESOS JUDICIALES,SISPROJ,175.0,SISPROJ-GAJ-GCAJ
320,322,ADQUISICION DE AMBULANCIAS URBANAS Y RURALES A...,SRCIAAUR-OCi,143.0,SRCIAAUR-OCi-OCI
321,323,Concurso Publico N 7-2022-ESSALUD/GCL-1 y proc...,SRCICPPCR-OCI,143.0,SRCICPPCR-OCI-OCI


In [7]:
# Exportar DataFrame a CSV
df.to_csv('resultado.csv', index=False)